# market_analy — Example Charts
Using DJI OHLC data from the test resources (no live API needed).

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib import dates as mdates
from market_analy import analysis
from market_analy.trends.analy import Trends, TrendsAlt
from market_analy.standalone import get_pct_off_high

BG  = '#1a1a2e'
TXT = '#e0e0e0'
GRN = '#00e676'
RED = '#ff5252'
BLU = '#64b5f6'
YEL = '#ffca28'
PUR = '#ce93d8'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'text.color': TXT, 'axes.labelcolor': TXT,
    'xtick.color': TXT, 'ytick.color': TXT,
    'axes.edgecolor': '#444466', 'grid.color': '#333355',
    'axes.grid': True, 'grid.alpha': 0.3,
})

# Load data
df = pd.read_csv('tests/resources/dji_1D.csv', index_col=0, parse_dates=True)
data_full = df.copy()
data = df.loc['2020':'2022'].copy()
data_2021 = df.loc['2021'].copy()

df15 = pd.read_csv('tests/resources/dji_15T.csv', index_col=0, parse_dates=True)
df15.columns = df15.columns.str.lower()
intra = df15.copy()

print(f'Daily data: {len(df)} bars  ({df.index[0].date()} to {df.index[-1].date()})')
print(f'15-min data: {len(intra)} bars  ({intra.index[0]} to {intra.index[-1]})')

## Chart 1 — Full DJI History: Price + 52-Week High + All-Time High

In [ ]:
from market_analy.standalone import get_ath, get_period_high

close = data_full['close']
ath   = get_ath(data_full['high'])         # all-time high series
ph52  = get_period_high(data_full['high'], window='365D')  # 52-week high

fig, axes = plt.subplots(3, 1, figsize=(14, 10),
                         gridspec_kw={'height_ratios': [3, 1.2, 1.2], 'hspace': 0.4})
ax1, ax2, ax3 = axes

# — Price + highs
ax1.plot(close.index, close.values, color=BLU, lw=1.1, label='DJI Close', zorder=3)
ax1.plot(ath.index, ath.values,   color=YEL, lw=0.9, ls='--', alpha=0.8, label='All-Time High')
ax1.plot(ph52.index, ph52.values, color=PUR, lw=0.8, ls=':', alpha=0.7, label='52-Week High')
ax1.set_title('DJI  |  Full History: Close vs ATH vs 52-Week High', fontsize=13, pad=10)
ax1.set_ylabel('Price (USD)')
ax1.legend(facecolor='#2a2a3e', fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# — % off ATH
pct_off_ath = (close / ath - 1) * 100
ax2.fill_between(close.index, pct_off_ath.values, 0, where=(pct_off_ath < 0),
                 color=RED, alpha=0.55, label='% off ATH')
ax2.plot(close.index, pct_off_ath.values, color=RED, lw=0.8)
ax2.axhline(0, color='#888888', lw=0.7, ls='--')
ax2.set_ylabel('% off ATH')
ax2.set_title('Drawdown from All-Time High', fontsize=10)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# — Volume
vols = data_full['volume'] / 1e6
bar_c = [GRN if c >= o else RED
         for o, c in zip(data_full['open'], data_full['close'])]
ax3.bar(data_full.index, vols, width=3, color=bar_c, alpha=0.65)
ax3.set_ylabel('Volume (M)')
ax3.set_title('Daily Volume', fontsize=10)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.savefig('/tmp/chart1_history.png', dpi=140, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /tmp/chart1_history.png')

## Chart 2 — Trend Detection: Advances & Declines (2020–2022)

In [ ]:
trends = Trends(data=data, interval='1D', prd=60,
                ext_break=0.05, ext_limit=0.03, min_bars=5)
movements = trends.get_movements()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9),
                                gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.35})

ax1.plot(data.index, data['close'], color=BLU, lw=1.4, label='DJI Close', zorder=3)

for m in movements:
    start = m.start
    end   = m.end if m.end is not None else data.index[-1]
    color = GRN if m.is_adv else RED
    ax1.axvspan(start, end, alpha=0.18, color=color, zorder=1)
    # label inside the shaded region
    mid   = start + (end - start) / 2
    mid_y = data['close'].loc[start:end].mean()
    ax1.annotate(f'{m.chg_pct:+.1%}\n{m.duration}d',
                 xy=(mid, mid_y), ha='center', va='center',
                 fontsize=8.5, color=color, fontweight='bold', zorder=5)
    # break line (dashed yellow)
    bl = m.line_break
    ax1.plot(bl.index, bl.values, color=YEL, lw=1.0, ls='--', alpha=0.8, zorder=2)
    # limit line (dotted purple)
    ll = m.line_limit
    ax1.plot(ll.index, ll.values, color=PUR, lw=0.8, ls=':', alpha=0.7, zorder=2)

# legend patches
handles = [
    plt.Line2D([0],[0], color=BLU, lw=1.5, label='DJI Close'),
    mpatches.Patch(color=GRN, alpha=0.5, label='Advance'),
    mpatches.Patch(color=RED, alpha=0.5, label='Decline'),
    plt.Line2D([0],[0], color=YEL, lw=1, ls='--', label='Break Line'),
    plt.Line2D([0],[0], color=PUR, lw=1, ls=':', label='Limit Line'),
]
ax1.legend(handles=handles, facecolor='#2a2a3e', fontsize=9, loc='upper left')
ax1.set_title('DJI 2020–2022  |  Trend Detection  (prd=60, ext_break=5%, ext_limit=3%)',
              fontsize=12, pad=10)
ax1.set_ylabel('Price (USD)')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.get_xticklabels(), visible=False)

bar_c = [GRN if c >= o else RED for o, c in zip(data['open'], data['close'])]
ax2.bar(data.index, data['volume'] / 1e6, width=1.5, color=bar_c, alpha=0.7)
ax2.set_ylabel('Vol (M)')
ax2.set_title('Volume', fontsize=10)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax2.get_xticklabels(), rotation=20, ha='right', fontsize=8)

plt.savefig('/tmp/chart2_trends.png', dpi=140, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /tmp/chart2_trends.png')

## Chart 3 — Rolling Drawdown & Yearly Return Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                          gridspec_kw={'width_ratios': [2.5, 1]})
ax1, ax2 = axes

# — % off 90-session rolling high across 2020-2022
poh = get_pct_off_high(data, window=90).iloc[:, 0]
ax1.fill_between(poh.index, poh.values, 0, where=(poh < 0),
                 color=RED, alpha=0.55)
ax1.plot(poh.index, poh.values, color=RED, lw=1.0)
ax1.axhline(0, color='#888888', lw=0.8, ls='--')
# annotate worst point
worst_dt = poh.idxmin()
worst_v  = poh.min()
ax1.annotate(f'{worst_v:.1f}%\n{worst_dt.date()}',
             xy=(worst_dt, worst_v), xytext=(worst_dt, worst_v - 1.5),
             color=RED, fontsize=8, ha='center',
             arrowprops=dict(arrowstyle='->', color=RED, lw=0.8))
ax1.set_title('% Off 90-Session Rolling High (2020–2022)', fontsize=11)
ax1.set_ylabel('% off rolling high')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.get_xticklabels(), rotation=20, ha='right', fontsize=8)

# — Monthly returns heatmap
monthly_ret = data['close'].resample('ME').last().pct_change() * 100
monthly_ret.index = monthly_ret.index.to_period('M')

years  = sorted(monthly_ret.index.year.unique())
months = list(range(1, 13))
grid   = np.full((len(years), 12), np.nan)
for yr_idx, yr in enumerate(years):
    for m in months:
        key = pd.Period(f'{yr}-{m:02d}', 'M')
        if key in monthly_ret.index:
            grid[yr_idx, m-1] = monthly_ret[key]

vmax = np.nanmax(np.abs(grid))
im = ax2.imshow(grid, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')
ax2.set_xticks(range(12))
ax2.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=8)
ax2.set_yticks(range(len(years)))
ax2.set_yticklabels(years, fontsize=9)
ax2.set_title('Monthly Returns Heatmap\n(green=positive, red=negative)', fontsize=10)
for (i, j), val in np.ndenumerate(grid):
    if not np.isnan(val):
        ax2.text(j, i, f'{val:+.1f}%', ha='center', va='center',
                 fontsize=6.5, color='white' if abs(val) > vmax*0.5 else '#333333')
fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04, label='% return')

plt.tight_layout()
plt.savefig('/tmp/chart3_drawdown_heatmap.png', dpi=140, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /tmp/chart3_drawdown_heatmap.png')

## Chart 4 — TrendsAlt on 15-min Intraday Data

In [ ]:
trends_alt = TrendsAlt(
    data=intra, interval='15min', prd=15,
    ext=0.005, rvr=0.5, rvr_init=0.8, grad=None, min_bars=3,
)
mvs_alt = trends_alt.get_movements()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.35})

ax1.plot(intra.index, intra['close'], color=BLU, lw=1.0, label='DJI 15-min')

for m in mvs_alt:
    start = m.start
    end   = m.end if m.end is not None else intra.index[-1]
    color = GRN if m.is_adv else RED
    ax1.axvspan(start, end, alpha=0.22, color=color, zorder=1)
    mid   = start + (end - start) / 2
    mid_y = intra['close'].loc[start:end].mean()
    ax1.text(mid, mid_y, f'{m.chg_pct:+.2%}',
             ha='center', va='center', fontsize=7.5,
             color=color, fontweight='bold', zorder=4)

handles = [
    plt.Line2D([0],[0], color=BLU, lw=1.5, label='DJI 15-min close'),
    mpatches.Patch(color=GRN, alpha=0.5, label='Advance'),
    mpatches.Patch(color=RED, alpha=0.5, label='Decline'),
]
ax1.legend(handles=handles, facecolor='#2a2a3e', fontsize=9)
ax1.set_title('DJI 15-min Intraday (May 2023)  |  TrendsAlt  (ext=0.5%, rvr=50%)',
              fontsize=12, pad=10)
ax1.set_ylabel('Price')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax1.get_xticklabels(), visible=False)

bar_c2 = [GRN if c >= o else RED for o, c in zip(intra['open'], intra['close'])]
ax2.bar(intra.index, intra['volume'] / 1e6, color=bar_c2, alpha=0.7)
ax2.set_ylabel('Vol (M)')
ax2.set_title('Volume', fontsize=10)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax2.get_xticklabels(), rotation=20, ha='right', fontsize=8)

plt.savefig('/tmp/chart4_intraday.png', dpi=140, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /tmp/chart4_intraday.png')

## Chart 5 — Movement Summary: Duration vs % Change scatter

In [ ]:
# Run Trends with tighter params to get more movements over full history
trends_all = Trends(data=data_full, interval='1D', prd=30,
                    ext_break=0.04, ext_limit=0.02, min_bars=5)
mvs_all = trends_all.get_movements()
print(f'Total movements over full history: {len(mvs_all)}')

adv_mvs = [m for m in mvs_all if m.is_adv]
dec_mvs = [m for m in mvs_all if m.is_dec]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax1, ax2  = axes

# — Scatter: duration vs pct change
adv_dur  = [m.duration for m in adv_mvs]
adv_chg  = [m.chg_pct * 100 for m in adv_mvs]
dec_dur  = [m.duration for m in dec_mvs]
dec_chg  = [m.chg_pct * 100 for m in dec_mvs]

ax1.scatter(adv_dur, adv_chg, color=GRN, alpha=0.7, s=60, label=f'Advances ({len(adv_mvs)})', edgecolors='none')
ax1.scatter(dec_dur, dec_chg, color=RED, alpha=0.7, s=60, label=f'Declines ({len(dec_mvs)})', edgecolors='none')
ax1.axhline(0, color='#888888', lw=0.8, ls='--')
ax1.set_xlabel('Duration (bars/sessions)')
ax1.set_ylabel('% Change')
ax1.set_title('Movements: Duration vs % Change\n(full DJI history, prd=30)', fontsize=11)
ax1.legend(facecolor='#2a2a3e', fontsize=9)

# — Distribution of % changes
bins = np.linspace(-50, 80, 30)
ax2.hist(adv_chg, bins=bins, color=GRN, alpha=0.65, label=f'Advances')
ax2.hist(dec_chg, bins=bins, color=RED, alpha=0.65, label=f'Declines')
ax2.axvline(np.mean(adv_chg), color=GRN, lw=1.5, ls='--',
            label=f'Adv mean: {np.mean(adv_chg):.1f}%')
ax2.axvline(np.mean(dec_chg), color=RED, lw=1.5, ls='--',
            label=f'Dec mean: {np.mean(dec_chg):.1f}%')
ax2.set_xlabel('% Change')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Movement Sizes', fontsize=11)
ax2.legend(facecolor='#2a2a3e', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/chart5_scatter.png', dpi=140, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /tmp/chart5_scatter.png')